In [1]:
import pyspark 

In [2]:
import os

python_path = r"C:\Users\94535\AppData\Local\Programs\Python\Python311\python.exe"

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Spark Session


In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .appName("crypto_streaming_processor") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.2,"
        "org.postgresql:postgresql:42.7.3,"
        "io.delta:delta-spark_4.1_2.13:4.1.0"
    ) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [ ]:
jdbc_url = (
    f"jdbc:postgresql://{os.environ['PGHOST']}:{os.getenv('PGPORT', '5432')}/{os.environ['PGDATABASE']}"
    "?ssl=true&sslmode=require"
)

In [ ]:
properties = {
    "user": os.environ["PGUSER"],
    "password": os.environ["PGPASSWORD"],
    "driver": "org.postgresql.Driver"
}

In [7]:
from pyspark.sql import functions as f 

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

In [9]:
from pyspark.sql.window import Window

# Data Schema 

In [10]:
schema = StructType([
    StructField("trade_id", StringType()),
    StructField("symbol", StringType()),
    StructField("price", DoubleType()),
    StructField("quantity", DoubleType()),
    StructField("volume", DoubleType()),
    StructField("event_time", LongType()),
    StructField("source", StringType())
])

# Ingesting data from Kafka

In [11]:
df_raw = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:29092").option("subscribe", "crypto_ticks").option("startingOffsets", "latest").load())

In [12]:
df_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



# Defining Schema

In [13]:
df_parsed = (
    df_raw
    .select(
        f.col("key").cast("string").alias("kafka_key"),
        f.from_json(f.col("value").cast("string"), schema).alias("data"),
        "topic",
        "partition",
        "offset",
        "timestamp"
    )
    .select(
        "data.*",
        "kafka_key",
        "topic",
        "partition",
        "offset",
        "timestamp"
    )
)

In [14]:
df_parsed.printSchema()

root
 |-- trade_id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- event_time: long (nullable = true)
 |-- source: string (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)



# Writing raw data 

In [15]:
df_parsed.writeStream.format("delta")\
    .partitionBy("symbol")\
    .option("header", "true")\
    .option("checkpointLocation", r"C:\Users\94535\Documents\rishikeshpandey\real-time-crypto-analytics\data_lake\checkpoint")\
    .outputMode("append")\
    .start(r"C:\Users\94535\Documents\rishikeshpandey\real-time-crypto-analytics\data_lake\streaming_output")

In [16]:
df_parsed.printSchema()

root
 |-- trade_id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- event_time: long (nullable = true)
 |-- source: string (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [17]:
df_parsed = df_parsed.withColumn("event_time", f.from_unixtime(f.col("event_time") / 1000).cast("timestamp"))

In [18]:
df_parsed.printSchema()

root
 |-- trade_id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- source: string (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [19]:
df_valid = df_parsed.select('trade_id').distinct()

In [20]:
df_invalid = df_parsed.groupBy("trade_id").agg(f.count(f.col("trade_id")).alias("count_tid")).filter(f.col('count_tid')>1)

In [21]:
df_invalid = df_parsed.filter((f.col("trade_id").isNull()) | (f.col("symbol").isNull()) | (f.col("price") < 0) | (f.col("quantity") <= 0) | (f.col("event_time").isNull()))

In [22]:
df_valid = df_parsed.filter((f.col("trade_id").isNotNull()) | (f.col("symbol").isNotNull()) | (f.col("price") >= 0) | (f.col("quantity") > 0) | (f.col("event_time").isNotNull()))

In [23]:
df_cleaned = df_valid.withColumn("processing_time",f.current_timestamp())

# Writing clean data

In [24]:
df_cleaned.writeStream.format("delta")\
    .option("header", "true")\
    .partitionBy("symbol")\
    .option("checkpointLocation", r"C:\Users\94535\Documents\rishikeshpandey\real-time-crypto-analytics\cleaned_layer\checkpoint")\
    .outputMode("append")\
    .start(r"C:\Users\94535\Documents\rishikeshpandey\real-time-crypto-analytics\cleaned_layer\streaming_output")

# Writing invalid data

In [25]:
df_invalid.writeStream.format("delta")\
    .option("header", "true")\
    .partitionBy("symbol")\
    .option("checkpointLocation", r"C:\Users\94535\Documents\rishikeshpandey\real-time-crypto-analytics\Invalid_data\checkpoint")\
    .outputMode("append")\
    .start(r"C:\Users\94535\Documents\rishikeshpandey\real-time-crypto-analytics\Invalid_data\streaming_output")

In [26]:
type(df_cleaned)

pyspark.sql.classic.dataframe.DataFrame

In [27]:
display(df_cleaned)

DataFrame[trade_id: string, symbol: string, price: double, quantity: double, volume: double, event_time: timestamp, source: string, kafka_key: string, topic: string, partition: int, offset: bigint, timestamp: timestamp, processing_time: timestamp]

In [28]:
df_agg = df_cleaned\
    .withWatermark("event_time","10 minutes")\
    .groupBy(f.col("symbol"), f.window("event_time","5 minutes"))\
    .agg(
        f.avg("price").alias("average_price"),
        f.max("price").alias("max_price"),
        f.min("price").alias("min_price"),
        f.sum("volume").alias("total_volume"),
        f.count("trade_id").alias("total_count")
    )

In [29]:
df_agg_final = df_agg.select(
    f.col("symbol"),
    f.col("window.start").alias("window_start"),
    f.col("window.end").alias("window_end"),
    f.col("average_price"),
    f.col("max_price"),
    f.col("min_price"),
    f.col("total_volume"),
    f.col("total_count")
)

In [30]:
df_agg_final.printSchema()

root
 |-- symbol: string (nullable = true)
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- average_price: double (nullable = true)
 |-- max_price: double (nullable = true)
 |-- min_price: double (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- total_count: long (nullable = false)



In [31]:
def write_market_summary_to_postgres(batch_df, batch_id):
    batch_df.write.jdbc(
        url=jdbc_url,
        table="crypto_market_summary_5min",
        mode="append",
        properties=properties
    )


In [32]:
query = df_agg_final.writeStream \
    .foreachBatch(write_market_summary_to_postgres) \
    .outputMode("update") \
    .option(
        "checkpointLocation",
        r"C:\Users\94535\Documents\rishikeshpandey\real-time-crypto-analytics\data_warehouse\crypto_market_summary_5min\checkpoint"
    ) \
    .start()

In [33]:
query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [34]:
print(query.isActive)


True


In [35]:
print(query.exception())


None


In [36]:
df_summary = spark.read.jdbc(
    url=jdbc_url,
    table="crypto_market_summary_5min",
    properties=properties
)

df_summary.show()


+-------+-------------------+-------------------+------------------+---------+---------+------------------+-----------+
| symbol|       window_start|         window_end|     average_price|max_price|min_price|      total_volume|total_count|
+-------+-------------------+-------------------+------------------+---------+---------+------------------+-----------+
|BTCUSDT|2026-06-20 16:55:00|2026-06-20 17:00:00|          63700.79| 63700.79| 63700.79|19.110236999999998|          1|
|ETHUSDT|2026-06-20 16:55:00|2026-06-20 17:00:00| 1726.863359375001|  1727.03|   1726.6|      64260.713014|        384|
|BTCUSDT|2026-06-20 17:10:00|2026-06-20 17:15:00| 63705.58150390656| 63713.61|  63702.0|230876.42858149976|       1024|
|BTCUSDT|2026-06-20 17:05:00|2026-06-20 17:10:00| 63700.74417136388|  63706.0| 63687.26| 414431.1417395999|       1774|
|BTCUSDT|2026-06-20 17:00:00|2026-06-20 17:05:00| 63697.61208307936| 63703.49| 63687.26| 740590.2429291982|       1637|
|ETHUSDT|2026-06-20 17:10:00|2026-06-20 